# NinaPro DB5 Exercise A 神经形态系统滑窗脉冲编码

本 Notebook 将 ninapro_data/processed/exerciseA/slide_window 中的训练集和测试集转换为神经形态电路脉冲数据。

处理约定：

- 使用逐通道 overall p1/p99 的 V3 分段线性映射，0 映射为 1.8639 V；
- 40 个原始采样各保持 4.999 ms，并用 1 μs 过渡，窗口总时长严格为 0.2 s；
- 电路与 neurophic_system_model/system_with_tia.py 一致；
- 脉冲采用 1.55 V 武装、下降穿越 1.25 V、0.5 ms 不应期；
- T 是 0.2 s 内的分箱数量，也是输入 SNN 的时间步数；
- 使用 CPU Numba 并行、批次断点续跑和 tqdm 进度条；
- 完整输出验证通过后生成 ninapro_data/spikes/T_{T}.zip。

输出 NPZ 字段：

- X：uint8 二值脉冲，形状为 [窗口数, 16, T]；
- spike_counts：uint16 分箱脉冲计数，形状为 [窗口数, 16, T]；
- Vin：float32 V3 映射电平，形状为 [窗口数, 16, 40]；
- y、subject、repetition、start_sample：原始窗口元数据。

远程完整运行前，将下一单元中的 RUN_FULL_ENCODING 改为 True。首次 Numba JIT 编译会产生一次性等待。


In [15]:
# 1. 参数、依赖与跨平台路径
T = 120
RUN_FULL_ENCODING = True

import csv
import importlib.util
import json
import math
import os
import shutil
import sys
import time
from collections import Counter
from dataclasses import asdict
from hashlib import sha256
from pathlib import Path, PurePosixPath
from zipfile import ZIP_DEFLATED, ZipFile

import numpy as np

try:
    import numba
    from numba import get_num_threads, njit, prange, set_num_threads
except ImportError as error:
    raise RuntimeError(
        "无法导入 Numba。请在当前内核中执行：python -m pip install -U numba tqdm"
    ) from error

try:
    from tqdm.auto import tqdm
except ImportError as error:
    raise RuntimeError(
        "无法导入 tqdm。请在当前内核中执行：python -m pip install -U tqdm"
    ) from error


def environment_flag(name, default=False):
    """读取常见形式的布尔环境变量。"""
    value = os.environ.get(name)
    if value is None:
        return default

    normalized = value.strip().lower()
    if normalized in {"1", "true", "yes", "on"}:
        return True
    if normalized in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"环境变量 {name} 必须是布尔值，实际为 {value!r}")


def find_project_root():
    """通过环境变量或当前目录向上寻找项目根目录。"""
    override = os.environ.get("NINAPRO_PROJECT_ROOT")
    start = Path(override).expanduser() if override else Path.cwd()
    start = start.resolve()

    for candidate in (start, *start.parents):
        metadata_path = (
            candidate
            / "ninapro_data"
            / "processed"
            / "exerciseA"
            / "slide_window"
            / "metadata.json"
        )
        model_path = (
            candidate
            / "neurophic_system_model"
            / "system_with_tia.py"
        )
        if metadata_path.is_file() and model_path.is_file():
            return candidate

    raise FileNotFoundError(
        "未找到项目根目录。请从项目目录启动 Jupyter，或设置 "
        "NINAPRO_PROJECT_ROOT。"
    )


if isinstance(T, (bool, np.bool_)) or not isinstance(T, (int, np.integer)):
    raise TypeError("T 必须是正整数")
if T <= 0:
    raise ValueError("T 必须是正整数")
T = int(T)

PROJECT_ROOT = find_project_root()
SOURCE_ROOT = (
    PROJECT_ROOT
    / "ninapro_data"
    / "processed"
    / "exerciseA"
    / "slide_window"
)
STATISTICS_PATH = (
    PROJECT_ROOT
    / "neurophic_system_model"
    / "ninapro_db5_exerciseA_channel_ranges"
    / "channel_statistics_overall.csv"
)
SYSTEM_MODEL_PATH = (
    PROJECT_ROOT
    / "neurophic_system_model"
    / "system_with_tia.py"
)
SPIKE_ROOT = PROJECT_ROOT / "ninapro_data" / "spikes"
VERSION_NAME = f"T_{T}"
OUTPUT_ROOT = SPIKE_ROOT / VERSION_NAME
BUILD_ROOT = SPIKE_ROOT / f".{VERSION_NAME}-building"
FINALIZE_ROOT = SPIKE_ROOT / f".{VERSION_NAME}-finalizing"
ARCHIVE_PATH = SPIKE_ROOT / f"{VERSION_NAME}.zip"

available_threads = get_num_threads()
default_threads = min(25, available_threads)
requested_threads = int(
    os.environ.get("NINAPRO_NUM_THREADS", str(default_threads))
)
NUM_THREADS = max(1, min(requested_threads, available_threads))
BATCH_SIZE = max(1, int(os.environ.get("NINAPRO_BATCH_SIZE", "128")))
OVERWRITE = environment_flag("NINAPRO_OVERWRITE", default=False)
ARCHIVE_OVERWRITE = environment_flag(
    "NINAPRO_ARCHIVE_OVERWRITE",
    default=False,
)

set_num_threads(NUM_THREADS)

print(f"项目目录：{PROJECT_ROOT}")
print(f"输入目录：{SOURCE_ROOT}")
print(f"输出目录：{OUTPUT_ROOT}")
print(f"压缩包：{ARCHIVE_PATH}")
print(f"T：{T}，单个分箱：{0.2 / T:.9f} s")
print(f"NumPy：{np.__version__}")
print(f"Numba：{numba.__version__}")
print(f"Numba CPU 线程：{get_num_threads()}")
print(f"窗口批大小：{BATCH_SIZE}")


项目目录：/root/autodl-tmp/NinaPro
输入目录：/root/autodl-tmp/NinaPro/ninapro_data/processed/exerciseA/slide_window
输出目录：/root/autodl-tmp/NinaPro/ninapro_data/spikes/T_120
压缩包：/root/autodl-tmp/NinaPro/ninapro_data/spikes/T_120.zip
T：120，单个分箱：0.001666667 s
NumPy：2.3.2
Numba：0.67.0
Numba CPU 线程：25
窗口批大小：128


In [16]:
# 2. 读取 V3 参数，并预计算 1 μs 过渡的公共 PWL 时间结构
SPLITS = ("train", "test")
EXPECTED_FIELDS = {"X", "y", "subject", "repetition", "start_sample"}
V0 = 1.8639
TRANSITION_SECONDS = 1e-6


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as file:
        return json.load(file)


def load_source_split(split_name):
    """完整读取一个较小的源 NPZ，并在关闭文件前复制数组。"""
    path = SOURCE_ROOT / f"{split_name}.npz"
    with np.load(path, allow_pickle=False) as loaded:
        if set(loaded.files) != EXPECTED_FIELDS:
            raise RuntimeError(
                f"{split_name}.npz 字段错误：{sorted(loaded.files)}"
            )
        return {name: loaded[name] for name in loaded.files}


def load_overall_percentiles(path, channel_count):
    """从统计 CSV 中读取每个通道的 overall p1 和 p99。"""
    rows = []
    with Path(path).open("r", encoding="utf-8-sig", newline="") as file:
        for row in csv.DictReader(file):
            if row["split"].strip().lower() == "overall":
                rows.append(row)

    rows.sort(key=lambda row: int(row["channel"]))
    channels = [int(row["channel"]) for row in rows]
    if channels != list(range(1, channel_count + 1)):
        raise RuntimeError(
            f"overall 统计通道应为 1..{channel_count}，实际为 {channels}"
        )

    p1 = np.asarray([float(row["p1"]) for row in rows], dtype=np.float64)
    p99 = np.asarray([float(row["p99"]) for row in rows], dtype=np.float64)
    if np.any(p1 >= 0.0) or np.any(p99 <= 0.0):
        raise RuntimeError("V3 要求所有通道满足 p1 < 0 < p99")
    return p1, p99


SOURCE_METADATA = read_json(SOURCE_ROOT / "metadata.json")
SAMPLE_RATE = float(SOURCE_METADATA["sample_rate_hz"])
CHANNEL_COUNT = int(SOURCE_METADATA["channels"])
WINDOW_SAMPLES = int(SOURCE_METADATA["window_samples"])
WINDOW_DURATION = WINDOW_SAMPLES / SAMPLE_RATE
MAXIMUM_VOLTAGE = 2.0 * V0

if SAMPLE_RATE != 200.0:
    raise RuntimeError(f"预期采样率 200 Hz，实际为 {SAMPLE_RATE}")
if CHANNEL_COUNT != 16:
    raise RuntimeError(f"预期 16 个通道，实际为 {CHANNEL_COUNT}")
if WINDOW_SAMPLES != 40:
    raise RuntimeError(f"预期窗口长度 40，实际为 {WINDOW_SAMPLES}")
if not math.isclose(WINDOW_DURATION, 0.2, rel_tol=0.0, abs_tol=1e-15):
    raise RuntimeError(f"预期窗口时长 0.2 s，实际为 {WINDOW_DURATION}")

P1, P99 = load_overall_percentiles(STATISTICS_PATH, CHANNEL_COUNT)


def map_to_vin(samples):
    """对形状 [..., 16, 时间] 的原始 EMG 应用 V3 映射。"""
    samples = np.asarray(samples, dtype=np.float64)
    if samples.ndim < 2 or samples.shape[-2] != CHANNEL_COUNT:
        raise ValueError(
            f"样本倒数第二维必须是 {CHANNEL_COUNT} 个通道：{samples.shape}"
        )

    reshape = (1,) * (samples.ndim - 2) + (CHANNEL_COUNT, 1)
    p1 = P1.reshape(reshape)
    p99 = P99.reshape(reshape)

    negative = V0 * (samples - p1) / (-p1)
    positive = V0 * (1.0 + samples / p99)
    voltage = np.where(samples < 0.0, negative, positive)
    return np.clip(voltage, 0.0, MAXIMUM_VOLTAGE)


def build_pwl_control_spec():
    """构造所有窗口共享的 PWL 时刻和控制点对应样本序号。"""
    sample_interval = 1.0 / SAMPLE_RATE
    if not 0.0 < TRANSITION_SECONDS < sample_interval:
        raise ValueError("过渡时间必须位于 0 和采样间隔之间")

    times = [0.0]
    sample_indices = [0]
    for index in range(WINDOW_SAMPLES):
        interval_end = (index + 1) * sample_interval
        times.append(interval_end - TRANSITION_SECONDS)
        sample_indices.append(index)
        times.append(interval_end)
        sample_indices.append(min(index + 1, WINDOW_SAMPLES - 1))

    times = np.asarray(times, dtype=np.float64)
    sample_indices = np.asarray(sample_indices, dtype=np.int16)
    if np.any(np.diff(times) <= 0.0):
        raise RuntimeError("PWL 控制点时间必须严格递增")
    if not math.isclose(
        float(times[-1]),
        WINDOW_DURATION,
        rel_tol=0.0,
        abs_tol=1e-15,
    ):
        raise RuntimeError("PWL 最后时刻不是窗口结束时刻")
    return times, sample_indices


def build_simulation_grid(pwl_time, maximum_step):
    """复现 system_with_tia.py 的规则，加入所有 PWL 拐点。"""
    step_count = max(int(math.ceil(WINDOW_DURATION / maximum_step)), 1)
    regular_time = np.linspace(
        0.0,
        WINDOW_DURATION,
        step_count + 1,
        dtype=np.float64,
    )
    breakpoints = pwl_time[
        (pwl_time > 0.0) & (pwl_time < WINDOW_DURATION)
    ]
    combined = np.sort(np.concatenate((regular_time, breakpoints)))
    tolerance = max(
        np.spacing(WINDOW_DURATION) * 8.0,
        maximum_step * 1e-10,
    )
    keep = np.r_[True, np.diff(combined) > tolerance]
    time_grid = combined[keep]
    time_grid[0] = 0.0
    time_grid[-1] = WINDOW_DURATION
    return time_grid


def build_grid_interpolation_spec(time_grid, pwl_time, sample_indices):
    """预计算每个仿真点由哪两个原始 Vin 电平线性插值得到。"""
    segment = np.searchsorted(pwl_time, time_grid, side="right") - 1
    segment = np.clip(segment, 0, len(pwl_time) - 2)
    left_time = pwl_time[segment]
    right_time = pwl_time[segment + 1]
    alpha = (time_grid - left_time) / (right_time - left_time)
    alpha = np.clip(alpha, 0.0, 1.0)
    return (
        np.ascontiguousarray(sample_indices[segment], dtype=np.int16),
        np.ascontiguousarray(sample_indices[segment + 1], dtype=np.int16),
        np.ascontiguousarray(alpha, dtype=np.float64),
    )


PWL_TIME, PWL_SAMPLE_INDEX = build_pwl_control_spec()

# 模型参数直接来自被要求复用的 Python 文件，避免手工参数漂移。
spec = importlib.util.spec_from_file_location(
    "ninapro_system_with_tia",
    SYSTEM_MODEL_PATH,
)
if spec is None or spec.loader is None:
    raise ImportError(f"无法加载模型：{SYSTEM_MODEL_PATH}")
system_with_tia = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = system_with_tia
spec.loader.exec_module(system_with_tia)

simulate_system = system_with_tia.simulate_system
PARAMETERS = system_with_tia._PARAMETERS
SIMULATION_TIME = build_simulation_grid(
    PWL_TIME,
    PARAMETERS.maximum_step,
)
(
    GRID_LEFT_SAMPLE,
    GRID_RIGHT_SAMPLE,
    GRID_ALPHA,
) = build_grid_interpolation_spec(
    SIMULATION_TIME,
    PWL_TIME,
    PWL_SAMPLE_INDEX,
)

print(f"V3 电压范围：0 ～ {MAXIMUM_VOLTAGE:.4f} V")
print(f"PWL 控制点：{len(PWL_TIME)}")
print(f"电路仿真点：{len(SIMULATION_TIME)}")
print(f"电路仿真区间：{len(SIMULATION_TIME) - 1}")


V3 电压范围：0 ～ 3.7278 V
PWL 控制点：81
电路仿真点：10041
电路仿真区间：10040


In [17]:
# 3. 原始 Python 电路参考路径，用于小规模逐元素一致性验证
def build_pwl_values(vin_sequence):
    """把 40 个 Vin 电平展开为与 PWL_TIME 对齐的控制点值。"""
    vin_sequence = np.asarray(vin_sequence, dtype=np.float64)
    if vin_sequence.shape != (WINDOW_SAMPLES,):
        raise ValueError(
            f"单通道 Vin 形状必须为 ({WINDOW_SAMPLES},)，实际为 "
            f"{vin_sequence.shape}"
        )
    return np.ascontiguousarray(vin_sequence[PWL_SAMPLE_INDEX])


def bin_reference_counts(spike_times):
    """按与 simulate_system 相同的边界规则构造 uint16 计数。"""
    spike_times = np.asarray(spike_times, dtype=np.float64)
    valid = spike_times[
        (spike_times >= 0.0) & (spike_times < WINDOW_DURATION)
    ]
    counts = np.zeros(T, dtype=np.uint16)
    if valid.size:
        indices = np.floor(valid * T / WINDOW_DURATION).astype(np.int64)
        np.add.at(counts, indices, 1)
    return counts


def reference_encode_vin(vin_sequence):
    """调用原始 Python 模型，返回二值脉冲和分箱计数。"""
    result = simulate_system(
        PWL_TIME,
        build_pwl_values(vin_sequence),
        T=T,
    )
    binary = np.asarray(result["spikes"], dtype=np.uint8)
    counts = bin_reference_counts(result["spike_times"])
    if not np.array_equal(binary, (counts > 0).astype(np.uint8)):
        raise AssertionError("参考实现的二值脉冲与计数不一致")
    return binary, counts


In [18]:
# 4. Numba 并行内核：融合电路推进、脉冲检测和分箱
N_VH = float(PARAMETERS.vh)
N_VL = float(PARAMETERS.vl)
N_RIN_NBOX = float(PARAMETERS.rin_nbox)
N_RME_NBOX = float(PARAMETERS.rme_nbox)
N_NBOX_RLOAD = float(PARAMETERS.nbox_rload)
N_CPARAL = float(PARAMETERS.cparal)

N_SYNAPSE_VTH = float(PARAMETERS.synapse_vth)
N_SYNAPSE_VNORM = float(PARAMETERS.synapse_vnorm)
N_SYNAPSE_PDRIVE = float(PARAMETERS.synapse_pdrive)
N_RON1 = float(PARAMETERS.ron1)
N_ROFF1 = float(PARAMETERS.roff1)
N_RON2 = float(PARAMETERS.ron2)
N_ROFF2 = float(PARAMETERS.roff2)
N_RON3 = float(PARAMETERS.ron3)
N_ROFF3 = float(PARAMETERS.roff3)
N_G11 = float(PARAMETERS.g11)
N_G12 = float(PARAMETERS.g12)
N_G13 = float(PARAMETERS.g13)
N_G23 = float(PARAMETERS.g23)
N_GV = float(PARAMETERS.gv)
N_SYNAPSE_VDS = float(PARAMETERS.synapse_vds)
N_STATE_CAPACITANCE = float(PARAMETERS.state_capacitance)

N_RF1 = float(PARAMETERS.rf1)
N_CF1 = float(PARAMETERS.cf1)
N_VREF = float(PARAMETERS.vref)
N_RIN2 = float(PARAMETERS.rin2)
N_RF2 = float(PARAMETERS.rf2)
N_NEGATIVE_RAIL = float(PARAMETERS.negative_rail)
N_POSITIVE_RAIL = float(PARAMETERS.positive_rail)

N_SPIKE_ARM = float(PARAMETERS.spike_arm_voltage)
N_SPIKE_FIRE = float(PARAMETERS.spike_fire_voltage)
N_REFRACTORY = float(PARAMETERS.spike_refractory)
N_FLOAT_TINY = float(np.finfo(np.float64).tiny)


@njit(inline="always", fastmath=False)
def advance_nbox_numba(voltage, high_resistance, input_voltage, dt):
    """严格复现原始解析 RC-NbOx 单步更新。"""
    nbox_resistance = (
        N_RIN_NBOX if high_resistance else N_RME_NBOX
    )
    target = (
        input_voltage
        * nbox_resistance
        / (N_NBOX_RLOAD + nbox_resistance)
    )
    tau = N_CPARAL / (
        1.0 / N_NBOX_RLOAD + 1.0 / nbox_resistance
    )
    next_voltage = target + (voltage - target) * math.exp(-dt / tau)

    crossed_high = high_resistance and next_voltage >= N_VH
    crossed_low = (not high_resistance) and next_voltage <= N_VL
    if not (crossed_high or crossed_low):
        return next_voltage, high_resistance

    threshold = N_VH if high_resistance else N_VL
    ratio = (threshold - target) / (voltage - target)
    ratio = min(max(ratio, N_FLOAT_TINY), 1.0)
    crossing_time = min(max(-tau * math.log(ratio), 0.0), dt)
    remaining_time = dt - crossing_time
    high_resistance = not high_resistance

    nbox_resistance = (
        N_RIN_NBOX if high_resistance else N_RME_NBOX
    )
    target = (
        input_voltage
        * nbox_resistance
        / (N_NBOX_RLOAD + nbox_resistance)
    )
    tau = N_CPARAL / (
        1.0 / N_NBOX_RLOAD + 1.0 / nbox_resistance
    )
    next_voltage = target + (threshold - target) * math.exp(
        -remaining_time / tau
    )
    return next_voltage, high_resistance


@njit(inline="always", fastmath=False)
def advance_state_numba(
    state,
    target,
    gate_is_on,
    on_resistance,
    off_resistance,
    dt,
):
    """复现单个突触状态的一阶解析更新。"""
    resistance = on_resistance if gate_is_on else off_resistance
    equilibrium = target if gate_is_on else 0.0
    decay = math.exp(-dt / (resistance * N_STATE_CAPACITANCE))
    return equilibrium + (state - equilibrium) * decay


@njit(parallel=True, fastmath=False)
def encode_vin_sequences_numba(
    vin_sequences,
    time_grid,
    left_sample,
    right_sample,
    alpha,
    bin_count,
):
    """将形状 [M, 40] 的 Vin 编码为二值脉冲和分箱计数。"""
    sequence_count = vin_sequences.shape[0]
    binary = np.zeros((sequence_count, bin_count), dtype=np.uint8)
    counts = np.zeros((sequence_count, bin_count), dtype=np.uint16)
    gain = N_RF2 / N_RIN2

    for sequence_index in prange(sequence_count):
        u1_high = True
        u2_high = True
        vout = 0.0
        final_out = 0.0
        vtia = 0.0
        state_x1 = 0.0
        state_x2 = 0.0
        state_x3 = 0.0
        armed = False
        last_spike_time = -np.inf

        left_index = left_sample[0]
        right_index = right_sample[0]
        weight = alpha[0]
        previous_vin = (
            (1.0 - weight) * vin_sequences[sequence_index, left_index]
            + weight * vin_sequences[sequence_index, right_index]
        )

        for step_index in range(1, time_grid.size):
            dt = time_grid[step_index] - time_grid[step_index - 1]
            left_index = left_sample[step_index]
            right_index = right_sample[step_index]
            weight = alpha[step_index]
            current_vin = (
                (1.0 - weight) * vin_sequences[sequence_index, left_index]
                + weight * vin_sequences[sequence_index, right_index]
            )
            interval_vin = 0.5 * (previous_vin + current_vin)
            previous_vin = current_vin

            vout, u1_high = advance_nbox_numba(
                vout,
                u1_high,
                interval_vin,
                dt,
            )

            normalized_gate = max(
                (vout - N_SYNAPSE_VTH)
                / (N_SYNAPSE_VNORM - N_SYNAPSE_VTH),
                0.0,
            ) ** N_SYNAPSE_PDRIVE
            gate_is_on = vout > N_SYNAPSE_VTH

            state_x1 = advance_state_numba(
                state_x1,
                normalized_gate,
                gate_is_on,
                N_RON1,
                N_ROFF1,
                dt,
            )
            state_x2 = advance_state_numba(
                state_x2,
                normalized_gate,
                gate_is_on,
                N_RON2,
                N_ROFF2,
                dt,
            )
            state_x3 = advance_state_numba(
                state_x3,
                normalized_gate,
                gate_is_on,
                N_RON3,
                N_ROFF3,
                dt,
            )

            conductance = max(
                N_G11 * state_x1 * state_x1
                + N_G12 * state_x1 * state_x2
                + N_G13 * state_x1 * state_x3
                + N_G23 * state_x2 * state_x3
                + N_GV * normalized_gate,
                0.0,
            )
            synapse_current = N_SYNAPSE_VDS * conductance

            tia_target = -N_RF1 * synapse_current
            tia_decay = math.exp(-dt / (N_RF1 * N_CF1))
            vtia = tia_target + (vtia - tia_target) * tia_decay
            vtia = min(max(vtia, N_NEGATIVE_RAIL), N_POSITIVE_RAIL)

            vdrive = (1.0 + gain) * N_VREF - gain * vtia
            vdrive = min(
                max(vdrive, N_NEGATIVE_RAIL),
                N_POSITIVE_RAIL,
            )

            previous_final = final_out
            final_out, u2_high = advance_nbox_numba(
                final_out,
                u2_high,
                vdrive,
                dt,
            )

            if final_out >= N_SPIKE_ARM:
                armed = True

            falling_crossing = (
                previous_final > N_SPIKE_FIRE
                and final_out <= N_SPIKE_FIRE
            )
            spike_time = time_grid[step_index]
            outside_refractory = (
                spike_time - last_spike_time >= N_REFRACTORY
            )

            if armed and falling_crossing and outside_refractory:
                last_spike_time = spike_time
                armed = False
                if 0.0 <= spike_time < WINDOW_DURATION:
                    bin_index = int(
                        math.floor(
                            spike_time * bin_count / WINDOW_DURATION
                        )
                    )
                    if 0 <= bin_index < bin_count:
                        binary[sequence_index, bin_index] = 1
                        counts[sequence_index, bin_index] += 1

    return binary, counts


def encode_vin_batch(vin_batch):
    """将 [B, 16, 40] Vin 批次编码为 [B, 16, T]。"""
    vin_batch = np.asarray(vin_batch, dtype=np.float64)
    expected_tail = (CHANNEL_COUNT, WINDOW_SAMPLES)
    if vin_batch.ndim != 3 or vin_batch.shape[1:] != expected_tail:
        raise ValueError(
            f"Vin 批次形状应为 [B, {CHANNEL_COUNT}, {WINDOW_SAMPLES}]，"
            f"实际为 {vin_batch.shape}"
        )
    if not np.all(np.isfinite(vin_batch)):
        raise ValueError("Vin 不能包含 NaN 或无穷大")

    batch_count = vin_batch.shape[0]
    sequences = np.ascontiguousarray(
        vin_batch.reshape(-1, WINDOW_SAMPLES),
        dtype=np.float64,
    )
    binary, counts = encode_vin_sequences_numba(
        sequences,
        SIMULATION_TIME,
        GRID_LEFT_SAMPLE,
        GRID_RIGHT_SAMPLE,
        GRID_ALPHA,
        T,
    )
    return (
        np.ascontiguousarray(
            binary.reshape(batch_count, CHANNEL_COUNT, T)
        ),
        np.ascontiguousarray(
            counts.reshape(batch_count, CHANNEL_COUNT, T)
        ),
    )


In [19]:
# 5. V3 锚点、原始 Python 一致性和吞吐量冒烟测试
anchor_samples = np.stack(
    (P1, np.zeros(CHANNEL_COUNT), P99),
    axis=-1,
)[None, ...]
anchor_voltage = map_to_vin(anchor_samples)[0]

np.testing.assert_allclose(anchor_voltage[:, 0], 0.0, atol=1e-12)
np.testing.assert_allclose(anchor_voltage[:, 1], V0, atol=1e-12)
np.testing.assert_allclose(
    anchor_voltage[:, 2],
    MAXIMUM_VOLTAGE,
    atol=1e-12,
)

if PWL_TIME.shape != (1 + 2 * WINDOW_SAMPLES,):
    raise AssertionError("PWL 控制点数量不正确")
if not math.isclose(PWL_TIME[-1], WINDOW_DURATION, abs_tol=1e-15):
    raise AssertionError("PWL 总时长不正确")

train_smoke = load_source_split("train")
test_smoke = load_source_split("test")
smoke_raw = np.stack((train_smoke["X"][0], test_smoke["X"][0]))
smoke_vin = map_to_vin(smoke_raw)
smoke_sequences = smoke_vin.reshape(-1, WINDOW_SAMPLES)

# 先触发 JIT，编译耗时不计入基准。
encode_vin_sequences_numba(
    np.ascontiguousarray(smoke_sequences[:1]),
    SIMULATION_TIME,
    GRID_LEFT_SAMPLE,
    GRID_RIGHT_SAMPLE,
    GRID_ALPHA,
    T,
)

selected_indices = np.asarray([0, 3, 15, 16 + 7, 16 + 15])
numba_binary, numba_counts = encode_vin_sequences_numba(
    np.ascontiguousarray(smoke_sequences[selected_indices]),
    SIMULATION_TIME,
    GRID_LEFT_SAMPLE,
    GRID_RIGHT_SAMPLE,
    GRID_ALPHA,
    T,
)

for result_index, sequence_index in enumerate(selected_indices):
    reference_binary, reference_counts = reference_encode_vin(
        smoke_sequences[sequence_index]
    )
    if not np.array_equal(
        reference_binary,
        numba_binary[result_index],
    ):
        mismatch = np.flatnonzero(
            reference_binary != numba_binary[result_index]
        )
        raise AssertionError(
            f"Numba 二值脉冲与参考实现不一致：sequence={sequence_index}, "
            f"bins={mismatch.tolist()}"
        )
    if not np.array_equal(
        reference_counts,
        numba_counts[result_index],
    ):
        mismatch = np.flatnonzero(
            reference_counts != numba_counts[result_index]
        )
        raise AssertionError(
            f"Numba 脉冲计数与参考实现不一致：sequence={sequence_index}, "
            f"bins={mismatch.tolist()}"
        )

benchmark_windows = min(BATCH_SIZE, 256, len(train_smoke["X"]))
benchmark_vin = map_to_vin(train_smoke["X"][:benchmark_windows])
benchmark_start = time.perf_counter()
benchmark_binary, benchmark_counts = encode_vin_batch(benchmark_vin)
benchmark_seconds = time.perf_counter() - benchmark_start

if not np.array_equal(
    benchmark_binary,
    (benchmark_counts > 0).astype(np.uint8),
):
    raise AssertionError("基准批次的二值脉冲与计数不一致")

total_windows = sum(
    int(SOURCE_METADATA["outputs"][name]["X_shape"][0])
    for name in SPLITS
)
estimated_current_seconds = (
    benchmark_seconds * total_windows / benchmark_windows
)
estimated_remote_seconds = (
    estimated_current_seconds * get_num_threads() / 25.0
)

SMOKE_TEST_RESULT = {
    "reference_sequences_checked": int(len(selected_indices)),
    "benchmark_windows": int(benchmark_windows),
    "benchmark_seconds": float(benchmark_seconds),
    "current_threads": int(get_num_threads()),
    "estimated_current_hours": estimated_current_seconds / 3600.0,
    "estimated_25_thread_hours": estimated_remote_seconds / 3600.0,
}

print("V3 三个锚点验证通过。")
print(
    f"{len(selected_indices)} 条通道序列与原始 Python 电路逐元素一致。"
)
print(
    f"{benchmark_windows} 个窗口耗时：{benchmark_seconds:.3f} s，"
    f"当前机器完整电路计算粗估：{estimated_current_seconds / 3600.0:.2f} h"
)
print(
    "仅按线程数线性折算到 25 vCPU："
    f"{estimated_remote_seconds / 3600.0:.2f} h；"
    "写盘、完整验证和 ZIP 压缩时间未计入；"
    "最终以远程 Notebook 实测基准为准。"
)


V3 三个锚点验证通过。
5 条通道序列与原始 Python 电路逐元素一致。
128 个窗口耗时：0.103 s，当前机器完整电路计算粗估：0.01 h
仅按线程数线性折算到 25 vCPU：0.01 h；写盘、完整验证和 ZIP 压缩时间未计入；最终以远程 Notebook 实测基准为准。


In [20]:
# 6. 分批断点续跑、最终 NPZ 校验和 ZIP 生成
ENCODER_VERSION = "ninapro-v3-python-circuit-numba-v1"
FINAL_NPZ_FIELDS = {
    "X",
    "spike_counts",
    "Vin",
    "y",
    "subject",
    "repetition",
    "start_sample",
}


def sha256_file(path, block_size=1024 * 1024):
    digest = sha256()
    with Path(path).open("rb") as file:
        for block in iter(lambda: file.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_write_json(path, value):
    path = Path(path)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as file:
        json.dump(value, file, ensure_ascii=False, indent=2)
        file.write("\n")
    temporary_path.replace(path)


def close_memmap(array):
    array.flush()
    memory_map = getattr(array, "_mmap", None)
    if memory_map is not None:
        memory_map.close()


def validate_safe_spike_path(path):
    """限制删除和移动操作只能作用于 ninapro_data/spikes 的子路径。"""
    path = Path(path).resolve()
    spike_root = SPIKE_ROOT.resolve()
    if path == spike_root or spike_root not in path.parents:
        raise ValueError(f"路径必须位于 {spike_root} 内：{path}")


def remove_temporary_directory(path):
    """只移除已通过范围校验的本任务临时目录。"""
    path = Path(path)
    validate_safe_spike_path(path)
    if path.exists():
        shutil.rmtree(path)


def build_configuration_payload():
    source_hashes = {
        f"{split_name}.npz": sha256_file(
            SOURCE_ROOT / f"{split_name}.npz"
        )
        for split_name in SPLITS
    }
    return {
        "encoder_version": ENCODER_VERSION,
        "dataset": "NinaPro DB5 Exercise A slide window",
        "dataset_scope": "full_train_and_test",
        "T": T,
        "window_duration_seconds": WINDOW_DURATION,
        "bin_duration_seconds": WINDOW_DURATION / T,
        "mapping": {
            "name": "V3 overall per-channel p1/p99",
            "zero_voltage": V0,
            "maximum_voltage": MAXIMUM_VOLTAGE,
            "channel_p1": P1.tolist(),
            "channel_p99": P99.tolist(),
            "statistics_sha256": sha256_file(STATISTICS_PATH),
        },
        "pwl": {
            "sample_rate_hz": SAMPLE_RATE,
            "sample_count": WINDOW_SAMPLES,
            "hold_seconds": 1.0 / SAMPLE_RATE - TRANSITION_SECONDS,
            "transition_seconds": TRANSITION_SECONDS,
            "duration_seconds": WINDOW_DURATION,
        },
        "spike_detection": {
            "arm_voltage": N_SPIKE_ARM,
            "fire_voltage": N_SPIKE_FIRE,
            "edge": "falling",
            "refractory_seconds": N_REFRACTORY,
            "binary_dtype": "uint8",
            "counts_dtype": "uint16",
        },
        "system_model_sha256": sha256_file(SYSTEM_MODEL_PATH),
        "source_sha256": source_hashes,
        "numpy_version": np.__version__,
        "numba_version": numba.__version__,
    }


def configuration_fingerprint(payload):
    serialized = json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    return sha256(serialized).hexdigest()


def initialize_or_resume_build(overwrite=False):
    SPIKE_ROOT.mkdir(parents=True, exist_ok=True)
    validate_safe_spike_path(OUTPUT_ROOT)
    validate_safe_spike_path(BUILD_ROOT)
    validate_safe_spike_path(FINALIZE_ROOT)

    payload = build_configuration_payload()
    fingerprint = configuration_fingerprint(payload)

    if overwrite:
        if OUTPUT_ROOT.exists():
            shutil.rmtree(OUTPUT_ROOT)
        remove_temporary_directory(BUILD_ROOT)
        remove_temporary_directory(FINALIZE_ROOT)
    elif OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()):
        raise FileExistsError(
            f"完整输出已经存在：{OUTPUT_ROOT}。"
            "如需重建，请设置 NINAPRO_OVERWRITE=1。"
        )

    progress_path = BUILD_ROOT / "progress.json"
    if BUILD_ROOT.exists():
        if not progress_path.is_file():
            raise RuntimeError(
                f"发现无法恢复的临时目录：{BUILD_ROOT}。"
                "确认后设置 NINAPRO_OVERWRITE=1 重建。"
            )
        progress = read_json(progress_path)
        if progress.get("fingerprint") != fingerprint:
            raise RuntimeError(
                "临时结果与当前 T、模型、映射、输入或依赖版本不一致，"
                "拒绝错误续算。请设置 NINAPRO_OVERWRITE=1 重建。"
            )
        print(f"继续已有进度：{progress['completed_rows']}")
        return progress, payload, fingerprint

    BUILD_ROOT.mkdir(parents=True, exist_ok=False)
    completed_rows = {}
    for split_name in SPLITS:
        row_count = int(
            SOURCE_METADATA["outputs"][split_name]["X_shape"][0]
        )
        split_root = BUILD_ROOT / split_name
        split_root.mkdir()

        arrays = (
            ("X.npy", np.uint8, (row_count, CHANNEL_COUNT, T)),
            (
                "spike_counts.npy",
                np.uint16,
                (row_count, CHANNEL_COUNT, T),
            ),
            (
                "Vin.npy",
                np.float32,
                (row_count, CHANNEL_COUNT, WINDOW_SAMPLES),
            ),
        )
        for filename, dtype, shape in arrays:
            output = np.lib.format.open_memmap(
                split_root / filename,
                mode="w+",
                dtype=dtype,
                shape=shape,
            )
            output[:] = 0
            close_memmap(output)
        completed_rows[split_name] = 0

    progress = {
        "fingerprint": fingerprint,
        "completed_rows": completed_rows,
    }
    atomic_write_json(progress_path, progress)
    return progress, payload, fingerprint


def validate_build_arrays():
    """分块验证临时 NPY，避免校验阶段额外复制全部数据。"""
    for split_name in SPLITS:
        row_count = int(
            SOURCE_METADATA["outputs"][split_name]["X_shape"][0]
        )
        split_root = BUILD_ROOT / split_name
        binary = np.load(
            split_root / "X.npy",
            mmap_mode="r",
            allow_pickle=False,
        )
        counts = np.load(
            split_root / "spike_counts.npy",
            mmap_mode="r",
            allow_pickle=False,
        )
        vin = np.load(
            split_root / "Vin.npy",
            mmap_mode="r",
            allow_pickle=False,
        )

        if binary.shape != (row_count, CHANNEL_COUNT, T):
            raise RuntimeError(f"{split_name} X 临时形状错误")
        if counts.shape != (row_count, CHANNEL_COUNT, T):
            raise RuntimeError(f"{split_name} spike_counts 临时形状错误")
        if vin.shape != (
            row_count,
            CHANNEL_COUNT,
            WINDOW_SAMPLES,
        ):
            raise RuntimeError(f"{split_name} Vin 临时形状错误")
        if binary.dtype != np.uint8:
            raise RuntimeError(f"{split_name} X 类型错误")
        if counts.dtype != np.uint16:
            raise RuntimeError(f"{split_name} spike_counts 类型错误")
        if vin.dtype != np.float32:
            raise RuntimeError(f"{split_name} Vin 类型错误")

        for start in range(0, row_count, 1024):
            end = min(start + 1024, row_count)
            binary_chunk = np.asarray(binary[start:end])
            counts_chunk = np.asarray(counts[start:end])
            vin_chunk = np.asarray(vin[start:end])
            if not np.array_equal(
                binary_chunk,
                (counts_chunk > 0).astype(np.uint8),
            ):
                raise RuntimeError(
                    f"{split_name} {start}:{end} 二值脉冲与计数不一致"
                )
            if not np.all(np.isfinite(vin_chunk)):
                raise RuntimeError(
                    f"{split_name} {start}:{end} Vin 非有限"
                )
            if np.any(vin_chunk < 0.0) or np.any(
                vin_chunk > MAXIMUM_VOLTAGE + 1e-6
            ):
                raise RuntimeError(
                    f"{split_name} {start}:{end} Vin 越界"
                )

        del binary, counts, vin


def build_final_metadata(payload, fingerprint, elapsed_seconds):
    source_counts = {}
    class_counts = {}
    output_shapes = {}
    total_spikes = {}

    for split_name in SPLITS:
        source = load_source_split(split_name)
        source_counts[split_name] = int(len(source["X"]))
        class_counts[split_name] = np.bincount(
            source["y"],
            minlength=12,
        ).astype(int).tolist()
        output_shapes[split_name] = {
            "X": [len(source["X"]), CHANNEL_COUNT, T],
            "spike_counts": [len(source["X"]), CHANNEL_COUNT, T],
            "Vin": [
                len(source["X"]),
                CHANNEL_COUNT,
                WINDOW_SAMPLES,
            ],
        }
        counts = np.load(
            BUILD_ROOT / split_name / "spike_counts.npy",
            mmap_mode="r",
            allow_pickle=False,
        )
        total_spikes[split_name] = int(np.sum(counts, dtype=np.uint64))
        del counts

    metadata = dict(payload)
    metadata.update(
        {
            "fingerprint": fingerprint,
            "complete": True,
            "source_counts": source_counts,
            "class_counts": class_counts,
            "output_shapes": output_shapes,
            "output_fields": {
                "X": "uint8 binary spikes",
                "spike_counts": "uint16 spike counts per bin",
                "Vin": "float32 compact V3 voltage levels",
                "y": "int64 model labels 0..11",
                "subject": "int16 subject IDs 1..10",
                "repetition": "int8 repetition IDs",
                "start_sample": "int32 source start indices",
            },
            "total_spikes": total_spikes,
            "elapsed_seconds_this_run": float(elapsed_seconds),
            "numba_threads": int(get_num_threads()),
            "batch_size": int(BATCH_SIZE),
        }
    )
    return metadata


def validate_final_output(root, metadata):
    """重新打开最终 NPZ，并验证完整字段、形状、类型和数据关系。"""
    root = Path(root)
    if read_json(root / "metadata.json") != metadata:
        raise RuntimeError("写盘后的 metadata.json 内容不一致")

    for split_name in SPLITS:
        source = load_source_split(split_name)
        expected_count = metadata["source_counts"][split_name]
        with np.load(
            root / f"{split_name}.npz",
            allow_pickle=False,
        ) as loaded:
            if set(loaded.files) != FINAL_NPZ_FIELDS:
                raise RuntimeError(
                    f"{split_name}.npz 字段错误：{sorted(loaded.files)}"
                )

            binary = loaded["X"]
            counts = loaded["spike_counts"]
            vin = loaded["Vin"]
            if binary.shape != (expected_count, CHANNEL_COUNT, T):
                raise RuntimeError(f"{split_name} X 形状错误")
            if counts.shape != (expected_count, CHANNEL_COUNT, T):
                raise RuntimeError(f"{split_name} spike_counts 形状错误")
            if vin.shape != (
                expected_count,
                CHANNEL_COUNT,
                WINDOW_SAMPLES,
            ):
                raise RuntimeError(f"{split_name} Vin 形状错误")
            if binary.dtype != np.uint8:
                raise RuntimeError(f"{split_name} X 类型错误")
            if counts.dtype != np.uint16:
                raise RuntimeError(f"{split_name} spike_counts 类型错误")
            if vin.dtype != np.float32:
                raise RuntimeError(f"{split_name} Vin 类型错误")
            if not np.array_equal(
                binary,
                (counts > 0).astype(np.uint8),
            ):
                raise RuntimeError(f"{split_name} 二值脉冲与计数不一致")
            if not np.all(np.isfinite(vin)):
                raise RuntimeError(f"{split_name} Vin 包含非有限值")
            if np.any(vin < 0.0) or np.any(
                vin > MAXIMUM_VOLTAGE + 1e-6
            ):
                raise RuntimeError(f"{split_name} Vin 越界")

            for name in (
                "y",
                "subject",
                "repetition",
                "start_sample",
            ):
                if not np.array_equal(loaded[name], source[name]):
                    raise RuntimeError(
                        f"{split_name} 的 {name} 未原样保留"
                    )


def finalize_build(payload, fingerprint, elapsed_seconds):
    validate_build_arrays()
    remove_temporary_directory(FINALIZE_ROOT)
    FINALIZE_ROOT.mkdir(parents=True, exist_ok=False)

    metadata = build_final_metadata(
        payload,
        fingerprint,
        elapsed_seconds,
    )
    try:
        for split_name in SPLITS:
            source = load_source_split(split_name)
            split_root = BUILD_ROOT / split_name
            binary = np.load(
                split_root / "X.npy",
                mmap_mode="r",
                allow_pickle=False,
            )
            counts = np.load(
                split_root / "spike_counts.npy",
                mmap_mode="r",
                allow_pickle=False,
            )
            vin = np.load(
                split_root / "Vin.npy",
                mmap_mode="r",
                allow_pickle=False,
            )
            np.savez_compressed(
                FINALIZE_ROOT / f"{split_name}.npz",
                X=binary,
                spike_counts=counts,
                Vin=vin,
                y=source["y"],
                subject=source["subject"],
                repetition=source["repetition"],
                start_sample=source["start_sample"],
            )
            del binary, counts, vin

        atomic_write_json(FINALIZE_ROOT / "metadata.json", metadata)
        validate_final_output(FINALIZE_ROOT, metadata)

        if OUTPUT_ROOT.exists():
            if any(OUTPUT_ROOT.iterdir()):
                raise RuntimeError(
                    f"最终输出目录意外为非空：{OUTPUT_ROOT}"
                )
            OUTPUT_ROOT.rmdir()
        FINALIZE_ROOT.replace(OUTPUT_ROOT)
        remove_temporary_directory(BUILD_ROOT)
    except Exception:
        remove_temporary_directory(FINALIZE_ROOT)
        raise

    print(f"完整数据已经写入：{OUTPUT_ROOT}")
    return metadata


def encode_full_dataset(overwrite=OVERWRITE):
    progress, payload, fingerprint = initialize_or_resume_build(overwrite)
    progress_path = BUILD_ROOT / "progress.json"
    started_at = time.perf_counter()

    total_windows = sum(
        int(SOURCE_METADATA["outputs"][name]["X_shape"][0])
        for name in SPLITS
    )
    completed_windows = sum(
        int(progress["completed_rows"][name])
        for name in SPLITS
    )

    with tqdm(
        total=total_windows,
        initial=completed_windows,
        desc="NinaPro 脉冲编码",
        unit="窗口",
        dynamic_ncols=True,
        mininterval=0.5,
        smoothing=0.1,
        leave=True,
    ) as progress_bar:
        for split_name in SPLITS:
            source = load_source_split(split_name)
            row_count = len(source["X"])
            start_row = int(progress["completed_rows"][split_name])
            if not 0 <= start_row <= row_count:
                raise RuntimeError(
                    f"{split_name} 断点越界：{start_row}/{row_count}"
                )

            progress_bar.set_postfix(
                split=split_name,
                split_progress=f"{start_row}/{row_count}",
                refresh=True,
            )
            if start_row == row_count:
                continue

            split_root = BUILD_ROOT / split_name
            output_binary = np.load(
                split_root / "X.npy",
                mmap_mode="r+",
                allow_pickle=False,
            )
            output_counts = np.load(
                split_root / "spike_counts.npy",
                mmap_mode="r+",
                allow_pickle=False,
            )
            output_vin = np.load(
                split_root / "Vin.npy",
                mmap_mode="r+",
                allow_pickle=False,
            )

            try:
                for batch_start in range(
                    start_row,
                    row_count,
                    BATCH_SIZE,
                ):
                    batch_end = min(
                        batch_start + BATCH_SIZE,
                        row_count,
                    )
                    raw_batch = source["X"][batch_start:batch_end]
                    vin_batch = map_to_vin(raw_batch)
                    binary_batch, counts_batch = encode_vin_batch(
                        vin_batch
                    )

                    output_binary[batch_start:batch_end] = binary_batch
                    output_counts[batch_start:batch_end] = counts_batch
                    output_vin[batch_start:batch_end] = vin_batch.astype(
                        np.float32
                    )
                    output_binary.flush()
                    output_counts.flush()
                    output_vin.flush()

                    # 先持久化断点，再更新进度条，显示值始终代表可恢复进度。
                    progress["completed_rows"][split_name] = batch_end
                    atomic_write_json(progress_path, progress)
                    progress_bar.update(batch_end - batch_start)
                    progress_bar.set_postfix(
                        split=split_name,
                        split_progress=f"{batch_end}/{row_count}",
                        refresh=False,
                    )
            finally:
                close_memmap(output_binary)
                close_memmap(output_counts)
                close_memmap(output_vin)

    elapsed_seconds = time.perf_counter() - started_at
    return finalize_build(payload, fingerprint, elapsed_seconds)


def create_spike_archive(
    source_dir=OUTPUT_ROOT,
    archive_path=ARCHIVE_PATH,
    overwrite=ARCHIVE_OVERWRITE,
):
    """验证最终输出后原子生成可提交到 GitHub 的 ZIP。"""
    source_dir = Path(source_dir).resolve()
    archive_path = Path(archive_path).resolve()
    validate_safe_spike_path(source_dir)
    validate_safe_spike_path(archive_path)

    metadata_path = source_dir / "metadata.json"
    if not metadata_path.is_file():
        raise FileNotFoundError(f"缺少编码元数据：{metadata_path}")
    metadata = read_json(metadata_path)
    if metadata.get("complete") is not True:
        raise RuntimeError("脉冲数据尚未完整生成，拒绝压缩")
    if int(metadata.get("T", -1)) != T:
        raise RuntimeError("输出 metadata 中的 T 与当前 Notebook 不一致")

    required_members = {
        f"{VERSION_NAME}/metadata.json",
        f"{VERSION_NAME}/train.npz",
        f"{VERSION_NAME}/test.npz",
    }
    source_files = (
        source_dir / "metadata.json",
        source_dir / "train.npz",
        source_dir / "test.npz",
    )
    missing = [path for path in source_files if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"缺少待压缩文件：{missing}")

    if archive_path.exists() and not overwrite:
        raise FileExistsError(
            f"ZIP 已存在：{archive_path}。"
            "如需重建，请设置 NINAPRO_ARCHIVE_OVERWRITE=1。"
        )

    temporary_path = archive_path.with_name(
        f".{archive_path.name}.tmp"
    )
    temporary_path.unlink(missing_ok=True)

    try:
        with ZipFile(
            temporary_path,
            mode="w",
            compression=ZIP_DEFLATED,
            compresslevel=9,
            allowZip64=True,
        ) as archive:
            for path in source_files:
                archive_name = (
                    PurePosixPath(VERSION_NAME) / path.name
                ).as_posix()
                archive.write(
                    path,
                    arcname=archive_name,
                    compress_type=ZIP_DEFLATED,
                    compresslevel=9,
                )

        with ZipFile(temporary_path, "r") as archive:
            names = {
                info.filename
                for info in archive.infolist()
                if not info.is_dir()
            }
            if names != required_members:
                raise RuntimeError(
                    f"ZIP 成员错误：{sorted(names)}"
                )
            bad_member = archive.testzip()
            if bad_member is not None:
                raise RuntimeError(
                    f"ZIP CRC 校验失败：{bad_member}"
                )

        temporary_path.replace(archive_path)
    except Exception:
        temporary_path.unlink(missing_ok=True)
        raise

    size_bytes = archive_path.stat().st_size
    archive_info = {
        "path": str(archive_path),
        "size_bytes": int(size_bytes),
        "sha256": sha256_file(archive_path),
        "members": sorted(required_members),
    }

    print(f"ZIP 已生成：{archive_path}")
    print(f"ZIP 大小：{size_bytes / 1024**2:.2f} MiB")
    print(f"SHA-256：{archive_info['sha256']}")
    if size_bytes > 100 * 1024**2:
        print("警告：超过 GitHub 普通 Git 单文件限制，请使用 Git LFS。")
    elif size_bytes > 25 * 1024**2:
        print("建议通过 Git 命令行提交，不要使用 GitHub 网页上传。")
    else:
        print("文件大小适合普通 Git 或 GitHub 网页上传。")
    return archive_info


In [21]:
# 7. 远程平台完整执行入口
if RUN_FULL_ENCODING:
    FULL_ENCODING_METADATA = encode_full_dataset(overwrite=OVERWRITE)
    ARCHIVE_INFORMATION = create_spike_archive(
        overwrite=ARCHIVE_OVERWRITE
    )
    print("完整训练集和测试集编码、验证及 ZIP 打包完成。")
else:
    FULL_ENCODING_METADATA = None
    ARCHIVE_INFORMATION = None
    print(
        "当前 RUN_FULL_ENCODING=False：已完成冒烟测试，未运行完整任务。"
        "在远程平台确认基准耗时后，将其改为 True 并重新运行本单元。"
    )


NinaPro 脉冲编码:   0%|          | 0/26626 [00:00<?, ?窗口/s]

完整数据已经写入：/root/autodl-tmp/NinaPro/ninapro_data/spikes/T_120
ZIP 已生成：/root/autodl-tmp/NinaPro/ninapro_data/spikes/T_120.zip
ZIP 大小：19.53 MiB
SHA-256：e5efe77a3dab12a3b04f787a331475e34840ddb05d5510838f29d62dccfbc3b4
文件大小适合普通 Git 或 GitHub 网页上传。
完整训练集和测试集编码、验证及 ZIP 打包完成。


## 远程运行提示

1. 建议在项目根目录启动 Jupyter。
2. 若缺少依赖，在当前内核执行 python -m pip install -U numba tqdm 后重启内核。
3. 保持 T 为所需 SNN 时间步数，并先运行到冒烟测试单元查看远程实测估时。
4. 确认后将 RUN_FULL_ENCODING=True，再运行完整执行入口。
5. 中断后使用相同的 T、输入、模型、NumPy 和 Numba 版本重新运行，会从 ninapro_data/spikes/.T_{T}-building 继续。
6. 完成后得到目录 ninapro_data/spikes/T_{T} 和压缩包 ninapro_data/spikes/T_{T}.zip。
